# Experiment matrix (root)

One-click root for the whole experiment. Runs the **k x i** matrix
(`experiment.outer_repeats` x `experiment.inner_repeats`).

- **k** = number of allocations: the machines are reserved `k` times. Every
  allocation selects its own machines, is provisioned, configured and validated
  on its own, runs the single `experiment.command` `i` times, and is torn down
  again.
- **i** = how often that one command runs per allocation.
- A **self-created** lease is created and deleted for every `k`; a
  **brought-along** lease (`reservation_id`/`reservation_ids`) is reused for
  all `k` allocations and kept -- only its servers are re-launched each time.

The logic lives in `lib/workflow.py` (single source of truth); this notebook
and `run.py` are only thin callers.

- **Phase 0** (once, interactive): prepare site/keypair and review the node
  selection rules (preview only).
- **Phase 1**: `run_matrix` -- per `k` select machines, provision/configure/
  validate, run the command `i` times + collect results, then tear the
  allocation down.

## Settings (edit here)

**This is the only section you need to edit:** choose the Chameleon site
(`CHI@UC` or `CHI@TACC`). The site selects its configuration file
(`configuration-uc.yaml` / `configuration-tacc.yaml`) with the matching
pgchecker image. Every cell below runs as-is.

In [ ]:
# Chameleon site to run on: "CHI@UC" or "CHI@TACC".
SITE = "CHI@UC"

## Setup

From here on, every cell runs as-is. This one loads the workflow library.

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd()))

import importlib
import lib.ssh, lib.chameleon, lib.workflow, lib.config
importlib.reload(lib.ssh)
importlib.reload(lib.chameleon)
importlib.reload(lib.config)
importlib.reload(lib.workflow)

from lib.config import load_config, experiment_plan
from lib.chameleon import (
    node_has_mellanox,
    node_has_at_most_ram_gib,
    get_node_name,
)
from lib.workflow import prepare_site_and_keypair, preview_node_selection, run_matrix

## Load configuration

In [ ]:
CONFIG_FILES = {
    "CHI@UC": "configuration-uc.yaml",
    "CHI@TACC": "configuration-tacc.yaml",
}
config = load_config(CONFIG_FILES[SITE])

name, outer_repeats, inner_repeats, command = experiment_plan(config)
print(f"Experiment: {name}  ->  k={outer_repeats} allocations x i={inner_repeats} runs = {outer_repeats * inner_repeats} runs")
print("Command:", command)
config

## Phase 0 - site/keypair + selection rules (preview)

Registers the keypair and shows what the rules below would pick. The preview is
**not binding** — `run_matrix` selects again for every `k`, and never the
machines the previous allocation used (a just-released machine is still being
cleaned, its deploy would fail with `status ERROR`). Only the directly preceding
allocation is skipped, so `k2` may run on `k0`'s machines again.

- `BLACKLIST` — machine names excluded from **every** allocation, for hosts that
  keep failing to deploy.
- `PREDICATE` / `SORT_KEY` — the hardware rules.

Excluded machines are never used as a fallback: if too few remain, `run_matrix`
raises and names what it skipped. The preview itself never raises — with a
brought-along lease it just reports that the lease already has its machines.

In [ ]:
prepare_site_and_keypair(config)

# Machines that are never selected, in any allocation, e.g. ["P3-CPU-010"].
BLACKLIST: list[str] = []

# Selection rules, applied per allocation. None/None = first available nodes.
PREDICATE = lambda n: node_has_mellanox(n) and node_has_at_most_ram_gib(n, 256)
SORT_KEY = lambda n: str(get_node_name(n) or "")

# Preview only -- run_matrix selects again per k.
preview_node_selection(
    config,
    predicate=PREDICATE,
    sort_key=SORT_KEY,
    blacklist=BLACKLIST,
)

## Phase 1 - run the matrix

For each `k` it selects machines afresh — never the ones the previous allocation
used — provisions the allocation, runs the command `i` times, collects the
results, and tears the allocation down again. A self-created lease is deleted per
`k`; a brought-along lease is kept. Results go to
`results/{experiment_id}/k{k}/i{i}/`, the machines used per allocation to
`results/{experiment_id}/k{k}/allocation.json`.

In [ ]:
from datetime import datetime, timezone

now = datetime.now(timezone.utc)
experiment_id = run_matrix(
    config,
    now=now,
    predicate=PREDICATE,
    sort_key=SORT_KEY,
    blacklist=BLACKLIST,
    on_step=print,
)
experiment_id

## Analysis

The raw results are under `results/{experiment_id}/k*/i*/`. To condense
them into best/worst overhead tables, run `AnalyzeBestWorstRuns.ipynb` with
(part of) this `experiment_id` as filter.

In [ ]:
import json

manifest_path = Path("results") / experiment_id / "manifest.json"
print(json.loads(manifest_path.read_text(encoding="utf-8")))